# Training API

Fine-tune forecasting models on your Lightning Rod datasets. This notebook walks through the full training workflow: generating a dataset, estimating cost, creating a training job, and monitoring progress.

The training API supports LoRA fine-tuning with configurable base models, training steps, batch size, and rank.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv openai

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Prepare the dataset

Training requires a dataset ID from a pipeline run. Run one of the other notebooks first to generate a dataset - each one prints the **Dataset ID** after `transforms.run()` — copy it into the cell below.

In [3]:
dataset_id = config.get_config_value("LIGHTNINGROD_DATASET_ID")

dataset = lr.datasets.get(dataset_id)
_ = dataset.download()


In [ ]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(days_to_resolution_range=(90, None)),
    split=SplitParams(test_size=0.2),
)

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [5]:
from lightningrod import GRPOTrainingConfig

config = GRPOTrainingConfig(
    base_model_id="Qwen/Qwen3-4B-Instruct-2507",
    training_steps=50,
)
cost_estimate = lr.training.estimate_cost(config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.02
Effective steps: 3
Train tokens: 63,349
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display.


In [6]:
job = lr.training.run(config, dataset=train_dataset, name="Forecasting fine-tune")
print(f"Job {job.id} completed with status: {job.status}")
print(f"Trained model ID: {job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job: Forecasting fine-tune                                                                                   │
│                                                                                                                 │
│    Reward: latest -0.3752  avg -0.8808  (3 steps)  (higher is better)                                           │
│                                                                                                                 │
│    Cost:  $0.01                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job 73184285-cd39-4afe-af8e-ceab345d80dc completed with status: COMPLETED
Trained model ID: checkpoint:73184285-cd39-4afe-af8e-ceab345d80dc


## Inference with your trained model

Use `lr.predict()` to run inference with your trained model. You can also use the OpenAI-compatible API directly — see [08_foresight_model.ipynb](08_foresight_model.ipynb) for the pre-trained foresight model.


In [19]:
print(lr.predict(job.model_id, "Will the Fed cut rates by 25bp in March 2026?"))


<answer>0.35</answer>


## Run evals on trained model

Run test evals on your trained model against a test dataset. The eval job runs the model on the dataset and reports metrics. Use the same dataset for a quick check, or a separate test split for production.

In [8]:
from lightningrod import EvalModel, training

eval_job = lr.evals.run(
    models=[
        EvalModel(model_id=config.base_model_id, label="Base"),
        EvalModel(model_id=job.model_id, label="Fine-tuned"),
    ],
    dataset=test_dataset,
)

training.print_eval(eval_job)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    ID: 4a78c3a1-a1a6-4288-9cd1-974066ddcc66                                                                     │
│    Model: checkpoint:73184285-cd39-4afe-af8e-ceab345d80dc                                                       │
│    Dataset: e87e04c3-4c0d-49ab-97bf-b30d724395d3                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┓                                                                    │
│  ┃ Metric              ┃    base ┃ trained ┃                                                                    │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━┩                                                                    │
│  │ brier_score         │  0.1925 │  0.1961 │                                                                    │
│  │ ece                 │  0.0671 │  0.0510 │                                                                    │
│  │ mean_reward         │ -0.5715 │ -0.5797 │                                                                    │
│  │ mean_valid_reward   │ -0.5715 │ -0.5797 │                                                                    │
│  │ n_samples           │      81 │      81 │                                                                    │
│  │ n_valid             │      81 │      81 │                                                                    │
│  │ parse_rate          │  1.0000 │  1.0000 │                                                                    │
│  │ total_cost          │  0.0016 │  0.0016 │                                                                    │
│  │ total_input_tokens  │   20154 │   20154 │                                                                    │
│  │ total_output_tokens │     787 │     787 │                                                                    │
│  └─────────────────────┴─────────┴─────────┘                                                                    │
│                                                                                                                 │
│    Cost:  $0.00                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

> Note: the trained model checkpoint will only be available for the period of 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.